# Onboarding A/B Test Analysis

**Experiment:** `onboarding_flow_v2`  
**Primary metric:** onboarding completion  
**Secondary / guardrail metrics:** trial start within 7 days, paid conversion within 14 days, D1/D7/D30 retention.

The analysis follows an **intention-to-treat (ITT)** approach: experiment groups are defined by assignment. Users with conflicting variant assignments are excluded upstream. Retention uses exact calendar-day `app_open` and only users with a complete observation window are eligible for each retention metric.

Statistical approach:
- 50/50 Sample Ratio Mismatch (SRM) check;
- two-sided two-proportion z-tests;
- 95% confidence intervals for treatment − control differences;
- raw p-value for the pre-defined primary metric;
- Holm correction across secondary metrics.


## 1. Load analytical dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from scipy.stats import chisquare, norm
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.multitest import multipletests

candidates = [
    Path('../data/ab_test_dataset.csv'),
    Path('data/ab_test_dataset.csv'),
    Path('ab_test_dataset.csv')
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('ab_test_dataset.csv not found. Export the SQL A/B dataset into data/.')

df = pd.read_csv(data_path)
print(f'Rows: {len(df):,}')
print(df['variant'].value_counts().sort_index())


Rows: 11,462
variant
control      5677
treatment    5785
Name: count, dtype: int64


## 2. Sample Ratio Mismatch check

In [2]:
variant_counts = df['variant'].value_counts()
control_n = int(variant_counts['control'])
treatment_n = int(variant_counts['treatment'])

observed = np.array([control_n, treatment_n])
expected = np.repeat(observed.sum() / 2, 2)
chi2_stat, srm_p_value = chisquare(observed, expected)

print(f'Control:   {control_n:,}')
print(f'Treatment: {treatment_n:,}')
print(f'Chi-square statistic: {chi2_stat:.4f}')
print(f'SRM p-value: {srm_p_value:.4f}')
print('No SRM detected.' if srm_p_value >= 0.05 else 'SRM detected — investigate allocation.')


Control:   5,677
Treatment: 5,785
Chi-square statistic: 1.0176
SRM p-value: 0.3131
No SRM detected.


## 3. Metric counts and denominators

In [3]:
metric_specs = [
    ('onboarding_completion', 'onboarding_completed', None),
    ('trial_start', 'trial_started', 'trial_eligible'),
    ('paid_conversion', 'paid_conversion', 'paid_eligible'),
    ('D1_retention', 'd1_retained', 'd1_eligible'),
    ('D7_retention', 'd7_retained', 'd7_eligible'),
    ('D30_retention', 'd30_retained', 'd30_eligible'),
]

rows = []
for metric, success_col, eligible_col in metric_specs:
    metric_df = df if eligible_col is None else df[df[eligible_col] == 1]
    for variant in ['control', 'treatment']:
        part = metric_df[metric_df['variant'] == variant]
        rows.append({
            'metric': metric,
            'variant': variant,
            'n': len(part),
            'success': int(part[success_col].sum())
        })

counts = pd.DataFrame(rows)
metric_counts = counts.pivot(index='metric', columns='variant', values=['n','success'])
metric_counts


n           success          
variant               control treatment control treatment
metric                                                   
D1_retention             5677      5785    2276      2393
D30_retention            5677      5785     823       869
D7_retention             5677      5785    1063      1041
onboarding_completion    5677      5785    3020      3271
paid_conversion          5677      5785     576       547
trial_start              5677      5785     889       989

## 4. Statistical test results

In [4]:
alpha = 0.05
results = []

for metric, success_col, eligible_col in metric_specs:
    metric_df = df if eligible_col is None else df[df[eligible_col] == 1]
    control = metric_df[metric_df['variant'] == 'control']
    treatment = metric_df[metric_df['variant'] == 'treatment']

    control_n = len(control)
    treatment_n = len(treatment)
    control_success = int(control[success_col].sum())
    treatment_success = int(treatment[success_col].sum())

    control_rate = control_success / control_n
    treatment_rate = treatment_success / treatment_n
    diff = treatment_rate - control_rate

    z_stat, p_value = proportions_ztest(
        count=np.array([treatment_success, control_success]),
        nobs=np.array([treatment_n, control_n]),
        alternative='two-sided'
    )

    se_diff = np.sqrt(
        treatment_rate * (1 - treatment_rate) / treatment_n +
        control_rate * (1 - control_rate) / control_n
    )
    z_crit = norm.ppf(0.975)
    ci_low = diff - z_crit * se_diff
    ci_high = diff + z_crit * se_diff

    results.append({
        'metric': metric,
        'control_n': control_n,
        'treatment_n': treatment_n,
        'control_rate_pct': control_rate * 100,
        'treatment_rate_pct': treatment_rate * 100,
        'difference_pp': diff * 100,
        'relative_uplift_pct': (diff / control_rate * 100) if control_rate > 0 else np.nan,
        'ci_low_pp': ci_low * 100,
        'ci_high_pp': ci_high * 100,
        'z_stat': z_stat,
        'p_value': p_value
    })

results_df = pd.DataFrame(results)
primary_metric = 'onboarding_completion'
primary_mask = results_df['metric'].eq(primary_metric)
secondary_mask = ~primary_mask

results_df['adjusted_p_value'] = results_df['p_value']
results_df['significant'] = False
results_df.loc[primary_mask, 'significant'] = results_df.loc[primary_mask, 'p_value'] < alpha

reject, corrected_p, _, _ = multipletests(
    results_df.loc[secondary_mask, 'p_value'], alpha=alpha, method='holm'
)
results_df.loc[secondary_mask, 'adjusted_p_value'] = corrected_p
results_df.loc[secondary_mask, 'significant'] = reject

results_df.round({
    'control_rate_pct': 2,
    'treatment_rate_pct': 2,
    'difference_pp': 2,
    'relative_uplift_pct': 2,
    'ci_low_pp': 2,
    'ci_high_pp': 2,
    'z_stat': 3,
    'p_value': 4,
    'adjusted_p_value': 4
})


,metric,control_n,treatment_n,control_rate_pct,treatment_rate_pct,difference_pp,relative_uplift_pct,ci_low_pp,ci_high_pp,z_stat,p_value,adjusted_p_value,significant
0,onboarding_completion,5677,5785,53.20,56.54,3.35,6.29,1.52,5.17,3.599,0.0003,0.0003,True
1,trial_start,5677,5785,15.66,17.10,1.44,9.17,0.08,2.79,2.077,0.0378,0.1890,False
2,paid_conversion,5677,5785,10.15,9.46,-0.69,-6.81,-1.78,0.40,-1.244,0.2136,0.6606,False
3,D1_retention,5677,5785,40.09,41.37,1.27,3.18,-0.52,3.07,1.388,0.1652,0.6606,False
4,D7_retention,5677,5785,18.72,17.99,-0.73,-3.90,-2.15,0.69,-1.009,0.3129,0.6606,False
5,D30_retention,5677,5785,14.50,15.02,0.52,3.62,-0.77,1.82,0.792,0.4287,0.6606,False


## 5. Paid-conversion guardrail sensitivity

The project brief does not define an acceptable non-inferiority margin for paid conversion. The table below is therefore a **sensitivity screen**, not a formal non-inferiority test. Safety is considered supported only when the full two-sided 95% CI stays above a hypothetical maximum acceptable decrease.


In [5]:
paid = results_df.loc[results_df['metric'] == 'paid_conversion'].iloc[0]
thresholds = [-0.25, -0.50, -1.00, -1.50, -2.00]

sensitivity = pd.DataFrame({'max_allowed_drop_pp': thresholds})
sensitivity['observed_effect_pp'] = paid['difference_pp']
sensitivity['ci_low_pp'] = paid['ci_low_pp']
sensitivity['ci_high_pp'] = paid['ci_high_pp']
sensitivity['guardrail_supported'] = sensitivity['ci_low_pp'] > sensitivity['max_allowed_drop_pp']
sensitivity['decision'] = np.where(
    sensitivity['guardrail_supported'], 'Safety supported', 'Insufficient evidence'
)

sensitivity.round(2)


,max_allowed_drop_pp,observed_effect_pp,ci_low_pp,ci_high_pp,guardrail_supported,decision
0,-0.25,-0.69,-1.78,0.4,False,Insufficient evidence
1,-0.50,-0.69,-1.78,0.4,False,Insufficient evidence
2,-1.00,-0.69,-1.78,0.4,False,Insufficient evidence
3,-1.50,-0.69,-1.78,0.4,False,Insufficient evidence
4,-2.00,-0.69,-1.78,0.4,True,Safety supported


## 6. Decision

In [6]:
primary = results_df.loc[results_df['metric'] == 'onboarding_completion'].iloc[0]
paid = results_df.loc[results_df['metric'] == 'paid_conversion'].iloc[0]

print(f"Primary metric: {primary['control_rate_pct']:.2f}% → {primary['treatment_rate_pct']:.2f}%")
print(f"Effect: {primary['difference_pp']:+.2f} pp; 95% CI [{primary['ci_low_pp']:+.2f}, {primary['ci_high_pp']:+.2f}]; p={primary['p_value']:.4f}")
print()
print(f"Paid conversion guardrail: {paid['control_rate_pct']:.2f}% → {paid['treatment_rate_pct']:.2f}%")
print(f"Effect: {paid['difference_pp']:+.2f} pp; 95% CI [{paid['ci_low_pp']:+.2f}, {paid['ci_high_pp']:+.2f}]; p={paid['p_value']:.4f}")
print()
print('Recommendation: DO NOT FULL ROLLOUT YET.')
print('The primary metric improved significantly, but downstream gains are not confirmed and paid-conversion safety is not established at tighter guardrail thresholds.')


Primary metric: 53.20% → 56.54%
Effect: +3.35 pp; 95% CI [+1.52, +5.17]; p=0.0003

Paid conversion guardrail: 10.15% → 9.46%
Effect: -0.69 pp; 95% CI [-1.78, +0.40]; p=0.2136

Recommendation: DO NOT FULL ROLLOUT YET.
The primary metric improved significantly, but downstream gains are not confirmed and paid-conversion safety is not established at tighter guardrail thresholds.


## 7. Export results

In [7]:
output_path = Path('../data/ab_test_results.csv')
if not output_path.parent.exists():
    output_path = Path('ab_test_results.csv')

results_df.to_csv(output_path, index=False)
print(f'Saved: {output_path}')


Saved: ../data/ab_test_results.csv
